# Training 11 · Perturbation-budget (ε) pilot

Chooses ε before any test data is seen: attackers with ε ∈ {0.5, 1, 2} against R0 seed 0, 2000 steps, scored on 400
BVCC **validation** utterances. The paper quotes the resulting ceiling rates (`pilot_ceiling_eps*`).
Three GPU jobs of about 10 minutes.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
import subprocess
repro.fetch(["r0_s0"])
r0 = repro.remap(repro.registry()["r0_s0"]["run_dir"] + "/results/selected_checkpoint.txt")
out_dir = repro.RUNS / "retrain" / "pilot_eps3"
out_dir.mkdir(parents=True, exist_ok=True)
for eps in ("0.5", "1.0", "2.0"):
    cmd = (f"python scripts/probe_attack.py --ckpt {r0} --config repro/configs/pilot_eps3_hack_loop.yaml "
           f"--eps {eps} --steps 2000 --n_val 400 --out {out_dir}/h0_e{eps}.json "
           f"--overrides action_hop=0 lr=1e-3 budget_penalty=0.0 budget_mode=sphere")
    if not LAUNCH:
        print(cmd)
    elif RUN == "local":
        subprocess.run(cmd, shell=True, check=True)
    else:
        script = (f"#!/bin/bash\n#SBATCH --account=PAS2966\n#SBATCH --partition=nextgen\n#SBATCH --time=01:00:00\n"
                  f"#SBATCH --gpus-per-node=1\n#SBATCH --cpus-per-task=12\n#SBATCH --mem=64G\n"
                  f"#SBATCH --output={out_dir}/%j.out\nset -euo pipefail\ncd {ROOT}\nsource scripts/env.sh\n{cmd}\n")
        print(subprocess.run(["sbatch", "--parsable"], input=script, capture_output=True, text=True).stdout.strip())

[fetch] 1 checkpoints ready in /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints


python scripts/probe_attack.py --ckpt /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt --config repro/configs/pilot_eps3_hack_loop.yaml --eps 0.5 --steps 2000 --n_val 400 --out /fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/pilot_eps3/h0_e0.5.json --overrides action_hop=0 lr=1e-3 budget_penalty=0.0 budget_mode=sphere
python scripts/probe_attack.py --ckpt /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt --config repro/configs/pilot_eps3_hack_loop.yaml --eps 1.0 --steps 2000 --n_val 400 --out /fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/pilot_eps3/h0_e1.0.json --overrides action_hop=0 lr=1e-3 budget_penalty=0.0 budget_mode=sphere
python scripts/probe_attack.py --ckpt /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt --config repro/configs/pilot_eps3_ha

## Retrained vs original

In [4]:
import json
rows = []
for eps in ("0.5", "1.0", "2.0"):
    for src, d in (("original", repro.fetch_results() / "pilot_eps3"), ("retrained", out_dir)):
        f = d / f"h0_e{eps}.json"
        if f.exists():
            r = json.loads(f.read_text())[0]
            rows.append({"eps": eps, "source": src, "val_ceiling_rate@0.3": r["val_ceiling_rate@0.3"],
                         "val_mean_gain": r["val_mean_gain"]})
pd.DataFrame(rows)

,eps,source,val_ceiling_rate@0.3,val_mean_gain
0,0.5,original,0.2225,0.090977
1,1.0,original,0.2150,0.071680
2,2.0,original,0.1850,0.052383
